In [80]:
import os, pickle, re
import numpy as np
import pandas as pd
import json

from physics.simulation import mcfm, msq
from physics.hstar import eft
from physics.variables import eft_terms, c6_degree, ct_degree, cg_degree
from nsbi import carl, taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [81]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amssymb}",
})

In [82]:
run_dir_4l = 'run/h4l'
run_dir_2l2v = 'run/h2l2v'

In [83]:
events_gg_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = taylr.utils.load_results(run_dir_4l, eft_terms)
events_gg_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = taylr.utils.load_results(run_dir_2l2v, eft_terms)

In [84]:
events_qq_4l, _, scaler_carl_4l, model_carl_4l = carl.utils.load_results(run_dir_4l)
events_qq_2l2v, _, scaler_carl_2l2v, model_carl_2l2v = carl.utils.load_results(run_dir_2l2v)

In [85]:
lumi = 3000.0

c6_linspace = [-25.0, 25.0, 101]
ct_linspace = [ -1.0,  1.0, 101]
cg_linspace = [ -1.0,  1.0, 101]
c6_space = np.linspace(*c6_linspace)
ct_space = np.linspace(*ct_linspace)
cg_space = np.linspace(*cg_linspace)

c6_sm = 0.0
cg_sm = 0.0
ct_sm = 0.0

i_c6_sm = np.where(np.isclose(c6_space, c6_sm))[0][0]
i_ct_sm = np.where(np.isclose(ct_space, ct_sm))[0][0]
i_cg_sm = np.where(np.isclose(cg_space, cg_sm))[0][0]

-10.0
0.0
-0.09999999999999998


In [86]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

sample_size = 50_000

events_gg_4l, events_gg_2l2v = events_gg_4l.sample(sample_size), events_gg_2l2v.filter(ZeroWeightFilter()).sample(sample_size)
events_qq_4l, events_qq_2l2v = events_qq_4l.sample(sample_size), events_qq_2l2v.filter(ZeroWeightFilter()).sample(sample_size)

w_gg_4l_sm, w_gg_2l2v_sm = events_gg_4l.weights.to_numpy(), events_gg_2l2v.weights.to_numpy()
w_qq_4l_sm, w_qq_2l2v_sm = events_qq_4l.weights.to_numpy(), events_qq_2l2v.weights.to_numpy()

sigma_gg_4l_sm = np.sum(w_gg_4l_sm)
sigma_gg_2l2v_sm = np.sum(w_gg_2l2v_sm)
sigma_qq_4l_sm = np.sum(w_qq_4l_sm)
sigma_qq_2l2v_sm = np.sum(w_qq_2l2v_sm)

events_4l = mcfm.stack(events_gg_4l, events_qq_4l)
events_2l2v = mcfm.stack(events_gg_2l2v, events_qq_2l2v)
sigma_4l_sm = events_4l.weights.sum()
sigma_2l2v_sm = events_2l2v.weights.sum()

events_gg = mcfm.stack(events_gg_4l, events_gg_2l2v)
events_qq = mcfm.stack(events_qq_4l, events_qq_2l2v)
sigma_gg_sm = events_gg.weights.sum()
sigma_qq = events_qq.weights.sum()

In [87]:
eft_mod = eft.Modifier(baseline=msq.Component.SBI, events=events_gg)

# (c6, ct)
w_gg_c6_ct_true, p_gg_c6_ct_true = eft_mod.modify(c6 = c6_space, ct = ct_space, cg = None)
sigma_gg_c6_ct_true = np.sum(w_gg_c6_ct_true, axis=0)  # c6 & cHbox
# w_gg_ct_cg_true, p_gg_c6_ct_true = w_gg_c6_ct_true[:,:,:,0], p_gg_c6_ct_true[:,:,:,0]

# (ct, cg)
w_gg_ct_cg_true, p_gg_ct_cg_true = eft_mod.modify(c6 = None, ct = ct_space, cg = cg_space)
sigma_gg_ct_cg_true = np.sum(w_gg_ct_cg_true, axis=0)  # c6 & cHbox
# w_gg_ct_cg_true, p_gg_ct_cg_true = w_gg_ct_cg_true[:,0,:,:], p_gg_ct_cg_true[:,0,:,:]

In [88]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
                'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
                'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
                'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 2048

In [89]:
trainer = L.Trainer(accelerator='gpu', devices=1)

kinematics_4l = events_4l.kinematics[features_4l]
X_carl_4l = scaler_carl_4l.transform(kinematics_4l.to_numpy())
dl_carl_4l = DataLoader(TensorDataset(torch.tensor(X_carl_4l, dtype=torch.float32)), batch_size=batch_size, num_workers=1) 
s_carl_4l = torch.cat(trainer.predict(model_carl_4l, dl_carl_4l))

kinematics_2l2v = events_2l2v.kinematics[features_2l2v].to_numpy()
X_carl_2l2v = scaler_carl_2l2v.transform(kinematics_2l2v)
dl_carl_2l2v = DataLoader(TensorDataset(torch.tensor(X_carl_2l2v, dtype=torch.float32)), batch_size=batch_size, num_workers=1) 
s_carl_2l2v = torch.cat(trainer.predict(model_carl_2l2v, dl_carl_2l2v))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [90]:
r_4l = s_carl_4l/(1-s_carl_4l)
r_2l2v = s_carl_2l2v/(1-s_carl_2l2v)
r = torch.cat((r_4l, r_2l2v))

In [91]:
X_taylr_4l = scaler_X_taylr_4l.transform(events_4l.kinematics[features_4l].to_numpy())
dl_taylr_4l = DataLoader(TensorDataset(torch.tensor(X_taylr_4l,dtype=torch.float32)), batch_size=batch_size)

X_taylr_2l2v = scaler_X_taylr_2l2v.transform(events_2l2v.kinematics[features_2l2v].to_numpy())
dl_taylr_2l2v = DataLoader(TensorDataset(torch.tensor(X_taylr_2l2v,dtype=torch.float32)), batch_size=batch_size)

coeffs_4l = [[[None for _ in range(cg_degree+1)] for _ in range(ct_degree+1)] for _ in range(c6_degree+1)]
coeffs_2l2v = [[[None for _ in range(cg_degree+1)] for _ in range(ct_degree+1)] for _ in range(c6_degree+1)]

for eft_term in eft_terms:
    i, j, k = eft_term

    output_4l = torch.cat(trainer.predict(models_taylr_4l[i][j][k], dl_taylr_4l))
    coeff_4l = scalers_y_taylr_4l[i][j][k].inverse_transform(output_4l.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_4l[i][j][k] = coeff_4l

    output_2l2v = torch.cat(trainer.predict(models_taylr_2l2v[i][j][k], dl_taylr_2l2v))
    coeff_2l2v = scalers_y_taylr_2l2v[i][j][k].inverse_transform(output_2l2v.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_2l2v[i][j][k] = coeff_2l2v

# populate (0,0,0) with 1
coeffs_4l[0][0][0] = np.ones_like(coeffs_4l[1][0][0])
coeffs_2l2v[0][0][0] = np.ones_like(coeffs_2l2v[1][0][0])

# populate any other None values with 0
for i in range(c6_degree+1):
    for j in range(ct_degree+1):
        for k in range(cg_degree+1):
            if coeffs_4l[i][j][k] is None:
                coeffs_4l[i][j][k] = np.zeros_like(coeffs_4l[0][0][0])
            if coeffs_2l2v[i][j][k] is None:
                coeffs_2l2v[i][j][k] = np.zeros_like(coeffs_2l2v[0][0][0])

# reshape to (Nevents, eft, ct, cg)
coeffs_4l = torch.tensor(coeffs_4l).permute(3,0,1,2)
coeffs_2l2v = torch.tensor(coeffs_2l2v).permute(3,0,1,2)
coeffs = torch.cat((coeffs_4l,coeffs_2l2v),dim=0)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [92]:
X_taylr_gg_4l = scaler_X_taylr_4l.transform(events_gg_4l.kinematics[features_4l].to_numpy())
dl_taylr_gg_4l = DataLoader(TensorDataset(torch.tensor(X_taylr_gg_4l,dtype=torch.float32)), batch_size=batch_size)

X_taylr_gg_2l2v = scaler_X_taylr_2l2v.transform(events_gg_2l2v.kinematics[features_2l2v].to_numpy())
dl_taylr_gg_2l2v = DataLoader(TensorDataset(torch.tensor(X_taylr_gg_2l2v,dtype=torch.float32)), batch_size=batch_size)

coeffs_gg_4l = [[[None for _ in range(cg_degree+1)] for _ in range(ct_degree+1)] for _ in range(c6_degree+1)]
coeffs_gg_2l2v = [[[None for _ in range(cg_degree+1)] for _ in range(ct_degree+1)] for _ in range(c6_degree+1)]

for eft_term in eft_terms:
    i, j, k = eft_term

    output_gg_4l = torch.cat(trainer.predict(models_taylr_4l[i][j][k], dl_taylr_gg_4l))
    coeff_gg_4l = scalers_y_taylr_4l[i][j][k].inverse_transform(output_gg_4l.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_gg_4l[i][j][k] = coeff_gg_4l

    output_gg_2l2v = torch.cat(trainer.predict(models_taylr_2l2v[i][j][k], dl_taylr_gg_2l2v))
    coeff_gg_2l2v = scalers_y_taylr_2l2v[i][j][k].inverse_transform(output_gg_2l2v.numpy().reshape(-1, 1)).reshape(-1)
    coeffs_gg_2l2v[i][j][k] = coeff_gg_2l2v

# populate (0,0,0) with 1
coeffs_gg_4l[0][0][0] = np.ones_like(coeffs_gg_4l[1][0][0])
coeffs_gg_2l2v[0][0][0] = np.ones_like(coeffs_gg_2l2v[1][0][0])

# populate any other None values with 0
for i in range(c6_degree+1):
    for j in range(ct_degree+1):
        for k in range(cg_degree+1):
            if coeffs_gg_4l[i][j][k] is None:
                coeffs_gg_4l[i][j][k] = np.zeros_like(coeffs_gg_4l[0][0][0])
            if coeffs_gg_2l2v[i][j][k] is None:
                coeffs_gg_2l2v[i][j][k] = np.zeros_like(coeffs_gg_2l2v[0][0][0])

# reshape to (Nevents, eft, ct, cg)
coeffs_gg_4l = torch.tensor(coeffs_gg_4l).permute(3,0,1,2)
coeffs_gg_2l2v = torch.tensor(coeffs_gg_2l2v).permute(3,0,1,2)
coeffs_gg = torch.cat((coeffs_gg_4l,coeffs_gg_2l2v),dim=0)
print(coeffs_gg.shape)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


...

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

torch.Size([100000, 5, 3, 3])


In [93]:
w_gg_sm = torch.tensor(events_gg.weights.to_numpy())
sigma_gg_sm = torch.sum(w_gg_sm)

w_gg_c6_ct = w_gg_sm[:,None,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg, c6_space, ct_space, None))
sigma_gg_c6_ct = torch.sum(w_gg_c6_ct, dim=0)  # c6 & cHbox

w_gg_ct_cg = w_gg_sm[:,None,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg, None, ct_space, cg_space))
sigma_gg_ct_cg = torch.sum(w_gg_ct_cg, dim=0)  # c6 & cHbox

w_gg_c6_cg = w_gg_sm[:,None,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg, c6_space, None, cg_space))
sigma_gg_c6_cg = torch.sum(w_gg_c6_cg, dim=0)  # c6 & cHbox

/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


In [94]:
# TEST: getting the ordering of events wrong should mess with everything below
# w_pp_sm = mcfm.stack(events_gg_2l2v,events_qq_4l,events_gg_4l,events_qq_2l2v).weights.to_numpy()

w_pp_sm = torch.tensor(mcfm.stack(events_gg_4l,events_qq_4l,events_gg_2l2v,events_qq_2l2v).weights.to_numpy())
sigma_pp_sm = torch.sum(w_pp_sm,dim=0)
p_pp_sm = w_pp_sm / sigma_pp_sm
nu_pp_sm = sigma_pp_sm * lumi

sigma_pp_c6_ct = sigma_gg_c6_ct + sigma_qq 
nu_pp_c6_ct = sigma_pp_c6_ct * lumi

sigma_pp_ct_cg = sigma_gg_ct_cg + sigma_qq 
nu_pp_ct_cg = sigma_pp_ct_cg * lumi

sigma_pp_c6_cg = sigma_gg_c6_cg + sigma_qq 
nu_pp_c6_cg = sigma_pp_c6_cg * lumi

In [95]:
eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)
eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

w_gg_4l_asimov, _ = eft_mod_gg_4l.modify(c6_asimov, ct_asimov, cg_asimov)
w_gg_2l2v_asimov, _ = eft_mod_gg_2l2v.modify(c6_asimov, ct_asimov, cg_asimov)

w_gg_4l_asimov = torch.tensor(w_gg_4l_asimov)
w_gg_2l2v_asimov = torch.tensor(w_gg_2l2v_asimov)

w_pp_asimov = torch.cat((w_gg_4l_asimov, torch.tensor(events_qq_4l.weights.to_numpy()), w_gg_2l2v_asimov, torch.tensor(events_qq_2l2v.weights.to_numpy())))
sigma_pp_asimov = torch.sum(w_pp_asimov, dim=0)
nu_pp_asimov = sigma_pp_asimov * lumi

print(w_pp_asimov/w_pp_sm)

tensor([1.0115, 1.0448, 1.0061,  ..., 1.0000, 1.0000, 1.0000],
       dtype=torch.float64)


In [96]:
# Poisson term
t_c6_ct_rate = -2 * nu_pp_asimov * (np.log(nu_pp_c6_ct) - np.log(nu_pp_sm)) + 2 * (nu_pp_c6_ct - nu_pp_sm) 
t_ct_cg_rate = -2 * nu_pp_asimov * (np.log(nu_pp_ct_cg) - np.log(nu_pp_sm)) + 2 * (nu_pp_ct_cg - nu_pp_sm) 
t_c6_cg_rate = -2 * nu_pp_asimov * (np.log(nu_pp_c6_cg) - np.log(nu_pp_sm)) + 2 * (nu_pp_c6_cg - nu_pp_sm) 

$$
\frac{p (x|\theta)}{p (x|0)} = \frac{\sigma_{gg}(0) + \sigma_{qq}}{\sigma_{gg}(\theta) + \sigma_{qq}} \frac{ \sigma_{gg}(0)\sum a_k(x) \theta^k + \sigma_{qq}\frac{s(x)}{1-s(x)} }{ \sigma_{gg}(0) + \sigma_{qq}\frac{s(x)}{1-s(x)} }
$$

In [97]:
carl_qq = sigma_qq * torch.tensor(r)[:,None,None]

taylr_gg_c6_ct = sigma_gg_sm * torch.tensor(eft.msq_eft_over_sm(coeffs, c6_space, ct_space, None))
pratio_pp_c6_ct =  (sigma_gg_sm + sigma_qq)/(sigma_gg_c6_ct+sigma_qq) * (taylr_gg_c6_ct + carl_qq)/(sigma_gg_sm + carl_qq)

taylr_gg_ct_cg = sigma_gg_sm * torch.tensor(eft.msq_eft_over_sm(coeffs, None, ct_space, cg_space))
pratio_pp_ct_cg =  (sigma_gg_sm + sigma_qq)/(sigma_gg_ct_cg+sigma_qq) * (taylr_gg_ct_cg + carl_qq)/(sigma_gg_sm + carl_qq)

taylr_gg_c6_cg = sigma_gg_sm * torch.tensor(eft.msq_eft_over_sm(coeffs, c6_space, None, cg_space))
pratio_pp_c6_cg =  (sigma_gg_sm + sigma_qq)/(sigma_gg_c6_cg+sigma_qq) * (taylr_gg_c6_cg + carl_qq)/(sigma_gg_sm + carl_qq)

/tmp/ipykernel_602952/1662349986.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl_qq = sigma_qq * torch.tensor(r)[:,None,None]
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


## Calibration of probability ratios

In [98]:
p_pp_sm = torch.tensor(p_pp_sm)
pratio_pp_c6_ct = torch.tensor(pratio_pp_c6_ct)
pratio_pp_ct_cg = torch.tensor(pratio_pp_ct_cg)
pratio_pp_c6_cg = torch.tensor(pratio_pp_c6_cg)

p_pp_c6_ct = pratio_pp_c6_ct * p_pp_sm[:, None, None]
psum_pp_c6_ct = torch.sum(p_pp_c6_ct, dim=0) 
pratio_c6_ct_calib = pratio_pp_c6_ct / psum_pp_c6_ct[None,:,:]

p_pp_ct_cg = pratio_pp_ct_cg * p_pp_sm[:, None, None]
psum_pp_ct_cg = torch.sum(p_pp_ct_cg, dim=0)
pratio_ct_cg_calib = pratio_pp_ct_cg / psum_pp_ct_cg[None,:,:]

p_pp_c6_cg = pratio_pp_c6_cg * p_pp_sm[:, None, None]
psum_pp_c6_cg = torch.sum(p_pp_c6_cg, dim=0)
pratio_c6_cg_calib = pratio_pp_c6_cg / psum_pp_c6_cg[None,:,:]

/tmp/ipykernel_602952/718663925.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  p_pp_sm = torch.tensor(p_pp_sm)
/tmp/ipykernel_602952/718663925.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pratio_pp_c6_ct = torch.tensor(pratio_pp_c6_ct)
/tmp/ipykernel_602952/718663925.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pratio_pp_ct_cg = torch.tensor(pratio_pp_ct_cg)
/tmp/ipykernel_602952/718663925.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().d

In [99]:
t_c6_ct_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_asimov)[:,None,None] * torch.log(pratio_c6_ct_calib) , dim=0)
t_ct_cg_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_asimov)[:,None,None] * torch.log(pratio_ct_cg_calib) , dim=0)
t_c6_cg_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_asimov)[:,None,None] * torch.log(pratio_c6_cg_calib) , dim=0)

/tmp/ipykernel_602952/1978476301.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_c6_ct_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_asimov)[:,None,None] * torch.log(pratio_c6_ct_calib) , dim=0)
/tmp/ipykernel_602952/1978476301.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_ct_cg_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_asimov)[:,None,None] * torch.log(pratio_ct_cg_calib) , dim=0)
/tmp/ipykernel_602952/1978476301.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_c6_cg_shape = - 2 * torch.sum(lumi * torch.tensor

In [100]:
t_rate = torch.tensor(t_c6_ct_rate)
t_shape = torch.tensor(t_c6_ct_shape)
t = t_rate + t_shape

cx_min, cx_max = -20, 25
cy_min, cy_max = -0.75, 0.5

cx_name = 'c6'
cy_name = 'ct'

cx_label = '$C_H$'
cy_label = '$C_{tH}$'

cx_linspace = c6_linspace
cy_linspace = ct_linspace

cx_space = c6_space
cy_space = ct_space

/tmp/ipykernel_602952/2352876752.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_rate = torch.tensor(t_c6_ct_rate)
/tmp/ipykernel_602952/2352876752.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_shape = torch.tensor(t_c6_ct_shape)


In [101]:
t_rate = torch.tensor(t_ct_cg_rate)
t_shape = torch.tensor(t_ct_cg_shape)
t = t_rate + t_shape

cx_min, cx_max = -0.75, 0.5
cy_min, cy_max = -1.0, 1.0

cx_name = 'ct'
cy_name = 'cg'

cx_label = '$C_{tH}$'
cy_label = '$C_{HG}$'

cx_linspace = ct_linspace
cy_linspace = cg_linspace

cx_space = ct_space
cy_space = cg_space

/tmp/ipykernel_602952/904772828.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_rate = torch.tensor(t_ct_cg_rate)
/tmp/ipykernel_602952/904772828.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_shape = torch.tensor(t_ct_cg_shape)


In [102]:
t_rate = torch.tensor(t_c6_cg_rate)
t_shape = torch.tensor(t_c6_cg_shape)
t = t_rate + t_shape

cx_min, cx_max = -20, 25
cy_min, cy_max = -1.0, 1.0

cx_name = 'c6'
cy_name = 'cg'

cx_label = '$C_{H}$'
cy_label = '$C_{HG}$'

cx_linspace = c6_linspace
cy_linspace = cg_linspace

cx_space = c6_space
cy_space = cg_space

/tmp/ipykernel_602952/3767733082.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_rate = torch.tensor(t_c6_cg_rate)
/tmp/ipykernel_602952/3767733082.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_shape = torch.tensor(t_c6_cg_shape)


In [103]:
i_cx_fit = torch.argmin(t) // cx_linspace[2]
i_cy_fit = torch.argmin(t)  % cy_linspace[2]

i_cx_fit_rate = torch.argmin(t_rate) // cx_linspace[2]
i_cy_fit_rate = torch.argmin(t_rate)  % cy_linspace[2]

t_min = t[i_cx_fit,i_cx_fit]
t_min_rate = t_rate[i_cx_fit_rate,i_cy_fit_rate]

X, Y = np.meshgrid(cx_space, cy_space)
Z = t.T
Z_rate = t_rate.T

print(cx_space[i_cx_fit], cy_space[i_cy_fit])

-10.0 -0.09999999999999998


In [ ]:
plt.figure(figsize=(6,4))

cl68 = 1.96

contours = plt.contour(X, Y, Z, levels=[t_min+1,t_min+4], colors='royalblue', linestyles=['--','-'])
contours_rate = plt.contour(X, Y, Z_rate, levels=[t_min_rate+1,t_min_rate+4], colors='grey', linestyles=['--','-'])

plt.scatter(cx_space[i_cx_fit], cy_space[i_cy_fit], marker='+', color='blue')

labels = [
    Line2D([0],[0],color='grey',label='$\\mathrm{Rate}$'),
    Line2D([0],[0],color='blue',label='$\\mathrm{NSBI}$'),
    Line2D([0],[0],color='black',linestyle='--',label='$68\\%\\ \\mathrm{CL}$'),
    Line2D([0],[0],color='black',linestyle='-',label='$95\\%\\ \\mathrm{CL}$'),
]
plt.legend(frameon=False, handles=labels, loc='upper center', ncols=2)

plt.xlabel(cx_label, fontsize=12)
plt.ylabel(cy_label, fontsize=12)

plt.xlim(cx_min, cx_max)
plt.ylim(cy_min, cy_max)

plt.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(f'nll_pp_zz_{cx_name}_{cy_name}_contours.pdf', bbox_inches='tight')

In [105]:
plt.figure(figsize=(6, 4))

# Heatmap of Z values
heatmap = plt.imshow(
    Z,
    extent=(cx_min, cx_max, cy_min, cy_max),
    origin='lower',
    aspect='auto',
    cmap='Blues',
    vmin=0.0,
    vmax=10.0
)

# Colorbar for reference
plt.colorbar(heatmap, label='NLL')

# Axis labels and formatting
plt.xlabel(cx_label, fontsize=12)
plt.ylabel(cy_label, fontsize=12)

plt.xlim(cx_min, cx_max)
plt.ylim(cy_min, cy_max)

plt.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(f'nll_pp_zz_{cx_name}_{cy_name}_full.pdf', bbox_inches='tight')
